## Install the package

First, clone the repo. In your terminal:

```bash
git clone https://github.com/EPPI-Centre/Flowde.git
cd Flowde
```

Install the package. In your terminal:

```bash
uv venv .venv --python 3.11
source .venv/bin/activate

uv pip install ipykernel jupyter ipywidgets
uv pip install -e ".[all]"

python -m ipykernel install --user --name flowde-env --display-name "Python (flowde-env)"
```

Then go into your notebook and select the `Python (flowde-env)` kernel.


## Download the Data

### Install git lfs

Check Git LFS is available. In your terminal:

```bash
git lfs version
```

If that command is not found, install Git LFS first. For windows, run the
following in your terminal:

```bash
winget install GitHub.GitLFS
```

### Install the data

Run this in your terminal in the `Flowde` directory:

```bash
git lfs install
git lfs pull --include="data/**"
```


## Extract Imgs


In [ ]:
from pathlib import Path

from flowde.extract_fns.paddle_layout_detect_extraction import (
    make_paddle_layout_extract_fn,
)
from flowde.extract_imgs import extract_imgs

PDF_DIR = Path("./../data/training-smoking-cessation/pdfs/")
EXTRACTED_IMGS_SAVE_DIR = Path(
    "./../data/training-smoking-cessation/extraction/my-extracted-images/"
)
N_JOBS = 1

In [ ]:
extract_fn = make_paddle_layout_extract_fn(device="cpu")

# Use gpu if you have installed gpu support:
# extract_fn = make_paddle_layout_extract_fn(device="gpu")

extract_imgs(
    pdf_dir=PDF_DIR,
    save_dir=EXTRACTED_IMGS_SAVE_DIR,
    extract_fn=extract_fn,
    n_jobs=N_JOBS,
)

## Classify Images


### Add a `.env`

At the root of your cloned repo, or in the same dir as this notebook, create a
`.env` file and add your OpenAI api key in the following format:

```text

OPENAI_API_KEY=your-openai-api-key

```


### Run classification


In [ ]:
from pathlib import Path

from flowde.classify_fns.openai_classify_fn import make_openai_classify_fn
from flowde.classify_fns.gemini_classify_fn import make_gemini_classify_fn
from flowde.classify_imgs import classify_imgs

IMGS_TO_CLASSIFY_DIR = Path(
    "../data/training-smoking-cessation/extraction/true_consort_pp_layout_detection_p3/"
)
MODEL = "gpt-5.5"
JSON_RESULTS_PATH = Path(
    "../data/training-smoking-cessation/classification/for-benchmark/classification-results-gpt-5.5.json"
)
POSITIVE_IMG_SAVE_DIR = Path(
    "../data/training-smoking-cessation/classification/positive-images/"
)

# MODEL = "gpt-5.5"  # Near perfect performance, but more expensive
MODEL_EFFORT = "high"
INPUT_TEXT = """
Classify whether this image is a flowchart.

Return 1 if it is a flowchart.
Return 0 if it is not a flowchart.
"""

In [ ]:
classify_fn = make_openai_classify_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
)

# classify_fn = make_gemini_classify_fn(
#     input_text=INPUT_TEXT,
#     model=MODEL,
#     effort=MODEL_EFFORT,
# )


labels = classify_imgs(
    classify_fn=classify_fn,
    img_dir=IMGS_TO_CLASSIFY_DIR,
    json_path=JSON_RESULTS_PATH,
    positive_img_save_dir=POSITIVE_IMG_SAVE_DIR,
    positive_classes={1},
)

## Rotate Images


In [ ]:
from pathlib import Path

from pydantic import BaseModel

from flowde.classify_fns.classify_types import RotationLabel
from flowde.classify_fns.openai_classify_fn import make_openai_classify_fn
from flowde.rotate_imgs import rotate_imgs

IMGS_TO_ROTATE_DIR = POSITIVE_IMG_SAVE_DIR
ROTATED_IMGS_SAVE_DIR = Path(
    "../data/training-smoking-cessation/rotation/rotation-corrected-images/"
)
JSON_ROTATIONS_RESULTS_PATH = Path(
    "../data/training-smoking-cessation/rotation/rotation-labels.json"
)

MODEL = "gpt-5.4-mini"  # Good performance but cheap
# MODEL = "gpt-5.5"  # Near perfect performance, but more expensive
MODEL_EFFORT = "high"
INPUT_TEXT = """
Return the clockwise angle required to correctly orient the image,
such that the majority of text reads left to right, top to bottom.
"""

In [ ]:
class RotationClassification(BaseModel):
    label: RotationLabel  # RotationLabel = Literal[0, 90, 180, 270]


classify_fn = make_openai_classify_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    result_structure=RotationClassification,
    effort=MODEL_EFFORT,
)

rotation_labels = rotate_imgs(
    classify_fn=classify_fn,
    img_dir=IMGS_TO_ROTATE_DIR,
    save_dir=ROTATED_IMGS_SAVE_DIR,
    json_path=JSON_ROTATIONS_RESULTS_PATH,
)

## Parse Images

For this step, if you want to get good results, it's important to use one of the
intelligent expensive models like `gpt5.5` and to parse the CONSORT diagrams in
parts


### Parse Nodes


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.gemini_parse import make_gemini_parse_fn
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

IMGS_TO_PARSE_DIR = Path(
    "../data/training-smoking-cessation/extraction/true_consort_pp_layout_detection_p3/"
)
PARSED_NODES_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/nodes")

# MODEL = "gpt-5.5"
MODEL = "gemini-3.1-pro-preview"
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (100, 150)
INPUT_TEXT = """
I am sending you an image of a participant flow diagram.

Your task is to parse only the node text and node numbers. Do not parse labels,
arrows, connections, additional text, or any other flow information.

## What counts as a node?

A node is a piece of text that represents a stage, event, count, or action in
the participant flow.

Nodes often:
- appear inside a box, circle, or other shape;
- If it has arrows pointing to it or away from it, consider it a node.
- describe how many participants were assessed, randomised, allocated, followed
  up, excluded, analysed, or lost;
- describe something that happened to participants, such as receiving an
  intervention or being included in an analysis.

A node does not have to be inside a shape. If text is essential to the
participant flow and is not being applied to multiple other nodes, it should still be parsed as a node.

Typical node examples include:
- "Assessed for eligibility (n = 250)"
- "Randomised (n = 120)"
- "Allocated to intervention (n = 60)"
- "Lost to follow-up (n = 5)"
- "Analysed (n = 55)"
- "Excluded from analysis (n = 3)"

## What should not be parsed as a node?

Do not parse section labels, stage labels, headings, or descriptive text that
applies to multiple nodes.

For example, text such as:
- "Enrollment"
- "Allocation"
- "Follow-up"
- "Analysis"
- "26 weeks"
- "52 weeks"

is usually a label, not a node, if it describes a row, column, branch, stage, or
group of nodes.

Do not parse these as nodes unless they are clearly part of the participant flow
itself.

The simplest explanation: Any piece of text that appears to apply to multiple nodes is a label, not a node.


## Text on arrows or between boxes

Sometimes important flow information is written on or beside an arrow rather
than inside a box.

Use judgement:
- If the text describes what happened to a specific group of participants, parse
  it as a node.
- If the text gives a participant count for exclusions, losses, follow-up, or
  analysis, parse it as a node.
- If the text is only a stage label, timing label, branch label, or heading that
  applies to several nearby nodes, do not parse it as a node.

For example:
- "Lost to follow-up (n = 8)" should usually be a node.
- "Did not receive intervention (n = 4)" should usually be a node.
- "Follow-up" should usually not be a node.
- "26 weeks" should usually not be a node.

## Duplicating nodes when branches join and split again

The goal is to represent the participant flow clearly in a structured format.

Sometimes two or more branches visually converge into one shared node, and then
split apart again from the converged node. In the image, the visual layout may make it clear which
outgoing branch belongs to which incoming branch, but this can become ambiguous
in a structured data format.

When this happens: duplicate it once for each branch. Each duplicated node should have the same text but a different node number.

Only duplicate a node when branches join and then diverge again from the box.
If two branches join to a box, but the branches do not split apart from that box again, do not duplicate the box.
If two branches point to a node, and the two branches continue, as long as the branches do not flow out from the converged node, you do not need to split the node.
Only duplicate a node if it is clear from the image that the two branches were intended to represent different groups of participants for the flow. If two branches join, into node A, but node A splits it into two branches that are not necessarily the same as the two branches entering, then you should not split the node. Split nodes are to clearly represent that the flow is separate groups.

If branches join and do not split again afterwards, do not duplicate the joined
node.

## Text transcription rules

Parse the text as it appears in the diagram.

- Preserve line breaks inside node text exactly as they appear in the diagram. Do not add new lines.  Do not remove new lines.
- Preserve meaningful punctuation.
- Preserve participant counts exactly.
- Always Preserve superscripts and subscripts using Unicode characters. You will often see ᵃ, ᵇ, ᶜ, ᵈ at the end of a word.
- Preseve uses of superscripts when use in cases like 1st, 2nd, 3rd, etc. But only if the specific example uses superscripts. If the diagram writes "1st" without a superscript, do not add a superscript.
- Preserve line break hyphenation as it appears in the diagram: hyphen and new line.
- Preserve bullet point symbols.
- Do not invent, simplify, or standardise text.
- Do not correct spelling in the diagram under any conditions
- Do not include text from labels, headings, arrows, captions, or additional
  notes unless that text is part of a node.
- Hyphenated words should not have spaces before or after the hyphen.

## Node numbering rules

Assign node numbers yourself.

- Node numbers must start at 1.
- Node numbers must increase by 1 for each node.
- Node numbers must be consecutive: 1, 2, 3, ...

## Exclusivity rule

Each piece of diagram text should belong to only one category.

If a piece of text is parsed as a node, it must not also be treated as a label
or additional text. If a piece of text is better understood as a label or
additional text, do not include it in the node output.

Return only the parsed nodes requested by the response schema.

SHORT CLARIFICATION ON THE NODES: Please take loosely my description on what a node is. It seems the more I try to explain sometimes the worse you get it. It is very obvious if you just look at the image, has the author intended this to be one of tthe boxes in a flowchart. If there are arrows pointing to it then almost certainly yes. If there aren't then almost certainly no.
"""


TRUE_NODES_DIR = Path("../data/training-smoking-cessation/parsing/ground-truth/nodes/")
TRUE_LABELS_DIR = Path(
    "../data/training-smoking-cessation/parsing/ground-truth/labels/"
)
TRUE_FLOW_DIR = Path("../data/training-smoking-cessation/parsing/ground-truth/flow/")
TRUE_ADDITIONAL_DIR = Path(
    "../data/training-smoking-cessation/parsing/ground-truth/additional_texts/"
)

In [ ]:
# parse_fn = make_openai_parse_fn(
#     input_text=INPUT_TEXT,
#     model=MODEL,
#     effort=MODEL_EFFORT,
#     parts_to_parse={"node_text"},
# )

parse_fn = make_gemini_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"node_text"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    save_dir=PARSED_NODES_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

In [ ]:
from flowde.benchmarks.parsing.parsing_bench import ParsingBenchmark
from flowde.benchmarks.parsing.text_distance_fns.levenshtein_fn import (
    levenshtein_with_nfc_and_space_normalisation,
)

benchmark = ParsingBenchmark(
    pred_diagrams_dir=PARSED_NODES_SAVE_DIR,
    distance_fn=levenshtein_with_nfc_and_space_normalisation,
    allow_missing_pred_diagrams=True,
    true_nodes_dir=TRUE_NODES_DIR,
    true_labels_dir=TRUE_LABELS_DIR,
    true_flow_dir=TRUE_FLOW_DIR,
    true_additional_texts_dir=TRUE_ADDITIONAL_DIR,
)

In [ ]:
import sys

size_bytes = sys.getsizeof(nml)
size_gb = size_bytes / 1024**3

print(size_gb)

In [ ]:
nml = benchmark.node_matches()[99:150]
nml = sorted(
    nml,
    key=lambda node_matches: node_matches.total_node_text_cost,
    reverse=True,
)
num_zero_cost = sum(node_matches.total_node_text_cost == 0 for node_matches in nml)
# print(f"Total node text edit: {benchmark.total_node_text_cost()}")
# print(
#     f"Avg diagram node text edit: {benchmark.total_node_text_cost() / len(benchmark.node_matches())}"
# )
print(f"Number of diagrams with zero total node text cost: {num_zero_cost}")

for node_matches in nml[:10]:
    print()
    print()
    print()
    print(f"Diagram: {node_matches.parent_img_code}")
    print(node_matches.total_node_text_cost)
    print()
    for match in node_matches.matches:
        if match.node_text_cost == 0:
            continue
        print()
        print()
        if match.true_node is not None:
            print(f"  True: {match.true_node.text}")
        else:
            print("  True: None")
        print()
        if match.pred_node is not None:
            print(f"  Pred: {match.pred_node.text}")
        else:
            print("  Pred: None")
        print()
        print(f"  Cost: {match.node_text_cost}")

### Parse Labels


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/labels")

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse labels
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"labels"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    save_dir=PARSED_LABELS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

### Parse Flow


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_DIR = PARSED_LABELS_SAVE_DIR
PARSED_FLOWS_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/flows")

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse flows
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"flow"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    labels_dir=PARSED_LABELS_DIR,
    save_dir=PARSED_FLOWS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

### Parse Additional Texts and combine into full CONSORT json


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_DIR = PARSED_LABELS_SAVE_DIR
PARSED_FLOWS_DIR = PARSED_FLOWS_SAVE_DIR
PARSED_FLOWCHARTS_SAVE_DIR = Path(
    "../data/training-smoking-cessation/parsing/pred/full-flowcharts"
)

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse additional text
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"node_text", "labels", "flow", "additional_texts"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    labels_dir=PARSED_LABELS_DIR,
    flow_dir=PARSED_FLOWS_DIR,
    save_dir=PARSED_FLOWCHARTS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)